In [0]:
import time, random
from datetime import datetime, timedelta, timezone
import numpy as np
from dataclasses import dataclass

In [0]:
def mean_reverting_step(
    current: float,
    target: float,
    alpha: float,
    noise_std: float,
    lower_bound: float,
    upper_bound: float,
    rng: np.random.Generator,
) -> float:
    """
    One discrete mean-reverting stochastic step.

    Parameters:
        current: current value
        target: target value
        alpha: mean reversion coefficient
        noise_std: standard deviation of the random noise
        lower_bound: lower bound of the value
        upper_bound: upper bound of the value
        rng: random number generator

    alpha controls the pull towards target:
        0.0 = no mean reversion
        1.0 = move all the way to target in one step
    """
    noise = rng.normal(loc=0.0, scale=noise_std)
    next_value = current + alpha * (target - current) + noise
    return float(np.clip(next_value, lower_bound, upper_bound))

In [0]:
@dataclass
class Buoy:
    buoy_id: str
    latitude: float
    longitude: float

buoys = [
    Buoy("PACIFIC_01", 0.0, 140.0),
    Buoy("PACIFIC_02", 0.0, 165.0),
    Buoy("PACIFIC_03", 0.0, -170.0),
    Buoy("PACIFIC_04", 0.0, -140.0),
    Buoy("PACIFIC_05", 0.0, -110.0),
]

In [0]:
def generate_stream(
    path: str,
    parameter_name: str,
    low: float,
    high: float,
    batch_interval: int,
    latency_max: int
):
    """
    Synthetic equatorial Pacific ocean-buoy simulator.

    Time model:
        One real-time interval (default: 60 seconds) advances simulation time by one simulated hour.

    Parameters:
        path: output directory
        parameter_name: name of the parameter to simulate (possible values: `temperature_c`, `salinity_psu`, `current_speed_ms`, `wind_speed_ms`, `wave_height_m`, `wave_period_s)
        low: lower bound of the parameter
        high: upper bound of the parameter
        batch_interval: number of seconds between each batch of data

    Output:
        One row per buoy per simulated hour, with both simulation time and
        ingestion time, so `device_count` rows each `batch_interval` seconds.

        Each row contains:
        - `buoy_id`: unique identifier for each simulated buoy
        - `latitude`: latitude of the simulated buoy
        - `longitude`: longitude of the simulated buoy
        - `simulation_timestamp`: UTC timestamp of the simulated hour
        - `ingestion_timestamp`: UTC timestamp of the actual generation time - random(0..latency_max) seconds
        - <parameter_name>: random value between low and high, calculated specifically for each column
    """

    rng = np.random.default_rng()
    span = high - low
    midpoint = (low + high) / 2.0

    # Per-buoy state so values evolve smoothly between batches (mean reversion)
    state = {buoy.buoy_id: midpoint for buoy in buoys}
    simulation_time = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)

    while True:
        now = datetime.now(timezone.utc)
        data = []
        for buoy in buoys:
            ts = now - timedelta(seconds=random.uniform(0, latency_max))
            current = state[buoy.buoy_id]

            # Daily cycle in local solar time (max ~15:00), shifted by buoy longitude
            local_solar_hour = (simulation_time.hour + buoy.longitude / 15.0) % 24.0
            daily_factor = float(np.cos(2.0 * np.pi * (local_solar_hour - 15.0) / 24.0))

            match parameter_name:
                case "temperature_c":
                    # Small daily heating cycle, low noise
                    target = midpoint + 0.05 * span * daily_factor
                    value = mean_reverting_step(current, target, 0.25, 0.01 * span, low, high, rng)
                case "salinity_psu":
                    # Very stable, tiny random variation
                    value = mean_reverting_step(current, midpoint, 0.30, 0.005 * span, low, high, rng)
                case "current_speed_ms":
                    # Slowly varying, weak pull to baseline
                    value = mean_reverting_step(current, midpoint, 0.10, 0.03 * span, low, high, rng)
                case "wind_speed_ms":
                    # Daily cycle plus occasional gusts (~10% of hours)
                    gust = rng.exponential(0.10 * span) if rng.random() < 0.10 else 0.0
                    target = midpoint + 0.10 * span * daily_factor + gust
                    value = mean_reverting_step(current, target, 0.30, 0.04 * span, low, high, rng)
                case "wave_height_m":
                    # Gradual response, moderate noise
                    value = mean_reverting_step(current, midpoint, 0.20, 0.03 * span, low, high, rng)
                case "wave_period_s":
                    # Governed by baseline swell, slow changes
                    value = mean_reverting_step(current, midpoint, 0.15, 0.02 * span, low, high, rng)
                case _:
                    # Unknown parameter: plain uniform random value
                    value = random.uniform(low, high)

            state[buoy.buoy_id] = value
            data.append({
                "buoy_id": buoy.buoy_id,
                "latitude": buoy.latitude,
                "longitude": buoy.longitude,
                "simulation_timestamp": simulation_time.isoformat(),
                "ingestion_timestamp": ts.isoformat(),
                parameter_name: round(value, 3),
            })

        df = spark.createDataFrame(data)
        df.write.mode("append").format("delta").save(path)
        simulation_time += timedelta(hours=1)
        time.sleep(batch_interval)

In [0]:
# Launch all 6 generators in parallel
from concurrent.futures import ThreadPoolExecutor

# common settings
batch_interval = 60
latency_max = 60

# (path, parameter_name, low, high)
streams = [
    ("/Volumes/ocean_monitoring/00_landing/streaming/water_temperature", "temperature_c", 24.0, 30.0),
    ("/Volumes/ocean_monitoring/00_landing/streaming/salinity", "salinity_psu", 33.0, 36.0),
    ("/Volumes/ocean_monitoring/00_landing/streaming/current_speed", "current_speed_ms", 0.0, 1.0),
    ("/Volumes/ocean_monitoring/00_landing/streaming/wind_speed/", "wind_speed_ms", 0.0, 10.0),
    ("/Volumes/ocean_monitoring/00_landing/streaming/wave_height/", "wave_height_m", 0.5, 2.5),
    ("/Volumes/ocean_monitoring/00_landing/streaming/wave_period/", "wave_period_s", 5.0, 12.0),
]

with ThreadPoolExecutor(max_workers=len(streams)) as executor:
    for path, parameter_name, low, high in streams:
        executor.submit(
            generate_stream, 
            path, 
            parameter_name, 
            low, 
            high,
            batch_interval, 
            latency_max
        )

    executor.shutdown(wait=True)